In [3]:
# Student name: Amir Gharghabi
# Course: ML
# HW4
# Q6

In [10]:
import pandas as pd
import numpy as np

In [5]:
# Assuming your datasets are in CSV format
train_data_path = 'dataset/adult.train.10k.discrete'
test_data_path = 'dataset/adult.test.10k.discrete'

# Load datasets into Pandas DataFrames
train_data = pd.read_csv(train_data_path)
test_data = pd.read_csv(test_data_path)

test_data

,<=50K,Private,11th,Never-married,Machine-op-inspct,Own-child,Black,Male,United-States
0,<=50K,Private,HS-grad,Married-civ-spouse,Farming-fishing,Husband,White,Male,United-States
1,>50K,Local-gov,Assoc-acdm,Married-civ-spouse,Protective-serv,Husband,White,Male,United-States
2,>50K,Private,Some-college,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,United-States
3,<=50K,Private,10th,Never-married,Other-service,Not-in-family,White,Male,United-States
4,>50K,Self-emp-not-inc,Prof-school,Married-civ-spouse,Prof-specialty,Husband,White,Male,United-States
...,...,...,...,...,...,...,...,...,...
9994,<=50K,Self-emp-not-inc,Prof-school,Married-civ-spouse,Exec-managerial,Husband,White,Male,United-States
9995,<=50K,Private,HS-grad,Never-married,Tech-support,Unmarried,White,Female,United-States
9996,<=50K,State-gov,HS-grad,Divorced,Protective-serv,Not-in-family,White,Male,United-States
9997,<=50K,Private,HS-grad,Married-civ-spouse,Exec-managerial,Husband,White,Male,United-States


In [6]:
train_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9999 entries, 0 to 9998
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   <=50K           9999 non-null   object
 1    State-gov      9999 non-null   object
 2    Bachelors      9999 non-null   object
 3    Never-married  9999 non-null   object
 4    Adm-clerical   9999 non-null   object
 5    Not-in-family  9999 non-null   object
 6    White          9999 non-null   object
 7    Male           9999 non-null   object
 8    United-States  9999 non-null   object
dtypes: object(9)
memory usage: 703.2+ KB


In [7]:
train_data.describe()

,<=50K,State-gov,Bachelors,Never-married,Adm-clerical,Not-in-family,White,Male,United-States
count,9999,9999,9999,9999,9999,9999,9999,9999,9999
unique,2,7,16,7,14,6,5,2,40
top,<=50K,Private,HS-grad,Married-civ-spouse,Prof-specialty,Husband,White,Male,United-States
freq,7549,7379,3279,4651,1327,4104,8601,6783,9090


In [20]:
class Node:
    def __init__(self, data=None, attribute=None, branches=None, label=None):
        self.data = data  # Data points at this node
        self.attribute = attribute  # Attribute to split on
        self.branches = branches  # Subtrees
        self.label = label  # Class label for leaf nodes

def id3(data, target_attribute, attributes):
    # Create a root node
    root = Node(data=data)

    # If all examples have the same class, return a leaf node
    if len(set(data[target_attribute])) == 1:
        root.label = data[target_attribute].iloc[0]
        return root

    # If there are no attributes left to split, return a leaf node with the majority class
    if not attributes:
        root.label = data[target_attribute].mode()[0]
        return root

    # Select the best attribute to split on
    best_attribute = choose_attribute(data, target_attribute, attributes)
    root.attribute = best_attribute

    # Split the dataset based on the selected attribute
    attribute_values = set(data[best_attribute])
    root.branches = {}
    for value in attribute_values:
        subset = data[data[best_attribute] == value].drop(columns=[best_attribute])
        root.branches[value] = id3(subset, target_attribute, [attr for attr in attributes if attr != best_attribute])

    return root

def choose_attribute(data, target_attribute, attributes):
    # Calculate the information gain for each attribute
    information_gains = {attr: calculate_information_gain(data, attr, target_attribute) for attr in attributes}
    
    # Choose the attribute with the highest information gain
    best_attribute = max(information_gains, key=information_gains.get)
    
    return best_attribute

def calculate_information_gain(data, attribute, target_attribute):
    # Calculate entropy before the split
    entropy_before = calculate_entropy(data[target_attribute])

    # Calculate weighted average entropy after the split
    attribute_values = set(data[attribute])
    entropy_after = 0
    for value in attribute_values:
        subset = data[data[attribute] == value]
        entropy_after += (len(subset) / len(data)) * calculate_entropy(subset[target_attribute])

    # Calculate information gain
    information_gain = entropy_before - entropy_after

    return information_gain

def calculate_entropy(labels):
    # Calculate entropy for a set of labels
    entropy = 0
    total = len(labels)

    for label in set(labels):
        p = len(labels[labels == label]) / total
        entropy -= p * np.log2(p)

    return entropy

In [21]:
target_attribute = '<=50K'
attributes = list(train_data.columns)
attributes.remove(target_attribute)

# Build the decision tree using ID3 algorithm
decision_tree = id3(train_data, target_attribute, attributes)

In [24]:
def predict(tree, instance):
    if tree.label is not None:
        return tree.label
    else:
        value = instance.get(tree.attribute)  # Use get to handle missing values
        if value in tree.branches:
            return predict(tree.branches[value], instance)
        else:
            # Handle unknown values by returning a default label or making a decision based on your specific requirements
            return tree.data[target_attribute].mode()[0]

def evaluate(tree, test_data):
    correct_predictions = 0
    total_instances = len(test_data)

    for _, instance in test_data.iterrows():
        predicted_label = predict(tree, instance)
        actual_label = instance[target_attribute]

        if predicted_label == actual_label:
            correct_predictions += 1

    accuracy = correct_predictions / total_instances
    return accuracy

In [26]:
# Evaluate the decision tree on the training data
train_accuracy = evaluate(decision_tree, train_data.fillna('default_value'))
print(f"Training Accuracy: {train_accuracy * 100:.2f}%")

# Evaluate the decision tree on the test data
test_accuracy = evaluate(decision_tree, test_data.fillna('default_value'))
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")

Training Accuracy: 87.54%
Test Accuracy: 75.39%
